# 02 tmdb metadata collection

This GitHub copy has been cleaned for readability: duplicate/empty cells were removed and saved cell outputs were cleared. Code and analysis workflow are retained.

**Execution note:** Original project paths may reference Google Drive/Colab. Update paths and provide required source files/API credentials in your own environment before rerunning.

# ============================================================
# Stage 2: TMDB Movie Metadata Collection
# Indian Cinema Trend Analytics
# ============================================================

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# ============================================================
# 02_tmdb_metadata_collection
# Setup & Configuration
# ============================================================

import os
import time
import requests
import pandas as pd

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/IndianCinemaTrendAnalytics"

INPUT_PATH = os.path.join(
    PROJECT_ROOT,
    "01_data",
    "processed",
    "ICMB_130_TMDB_MAPPING.csv"
)

OUTPUT_DIR = os.path.join(
    PROJECT_ROOT,
    "01_data",
    "raw"
)

# Create output directory if it does not exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Output file for raw TMDB metadata
OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_TMDB_METADATA.json"
)

# ------------------------------------------------------------
# TMDB API configuration
# ------------------------------------------------------------

BASE_URL = "https://api.themoviedb.org/3"

# Use the API key variable you already configured earlier.
# If your config.py contains TMDB_API_KEY, this will use it.
try:
    from config import TMDB_API_KEY
except ImportError:
    TMDB_API_KEY = None

# ------------------------------------------------------------
# Basic settings
# ------------------------------------------------------------

REQUEST_DELAY = 0.25
MAX_RETRIES = 3
TIMEOUT = 30

print("Project root:", PROJECT_ROOT)
print("Input file:", INPUT_PATH)
print("Input exists:", os.path.exists(INPUT_PATH))
print("Output directory:", OUTPUT_DIR)
print("Output directory exists:", os.path.exists(OUTPUT_DIR))
print("TMDB API key loaded:", TMDB_API_KEY is not None)

In [ ]:
from pathlib import Path

CONFIG_DIR = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/04_config"
)

CONFIG_DIR.mkdir(parents=True, exist_ok=True)

config_content = '''
# ============================================================
# Indian Cinema Trend Analytics
# TMDB API Configuration
# ============================================================

# TMDB API Key
# Replace the value below with your actual TMDB API key.
TMDB_API_KEY = "YOUR_TMDB_API_KEY"

# TMDB API base URL
BASE_URL = "https://api.themoviedb.org/3"

# API language
LANGUAGE = "en-US"

# Request settings
REQUEST_DELAY = 0.25
MAX_RETRIES = 3
TIMEOUT = 30

# Project settings
START_YEAR = 2000
END_YEAR = 2025
'''

config_path = CONFIG_DIR / "config.py"

with open(config_path, "w", encoding="utf-8") as f:
    f.write(config_content)

print("Config file created:")
print(config_path)
print("Exists:", config_path.exists())

In [ ]:
# ============================================================
# Load Verified TMDB Mapping
# ============================================================

import os
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/IndianCinemaTrendAnalytics"

INPUT_PATH = os.path.join(
    PROJECT_ROOT,
    "01_data",
    "processed",
    "ICMB_130_TMDB_MAPPING.csv"
)

RAW_DIR = os.path.join(
    PROJECT_ROOT,
    "01_data",
    "raw"
)

print("Loading verified TMDB mapping...")
print("Input:", INPUT_PATH)

# Check file
if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(
        f"TMDB mapping file not found:\n{INPUT_PATH}"
    )

# Load mapping
tmdb_mapping = pd.read_csv(INPUT_PATH)

print("\nTMDB mapping loaded successfully.")
print("Rows:", len(tmdb_mapping))
print("Columns:", len(tmdb_mapping.columns))

# Basic validation
required_columns = [
    "movie_record_id",
    "dataset_title",
    "dataset_year",
    "dataset_language",
    "tmdb_id",
    "tmdb_title",
    "tmdb_release_date",
    "tmdb_original_language"
]

missing_columns = [
    col for col in required_columns
    if col not in tmdb_mapping.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# Validate IDs
print("\nValidation:")
print("Unique movie records:", tmdb_mapping["movie_record_id"].nunique())
print("Unique TMDB IDs:", tmdb_mapping["tmdb_id"].nunique())
print("Missing TMDB IDs:", tmdb_mapping["tmdb_id"].isna().sum())
print("Duplicate movie records:",
      tmdb_mapping["movie_record_id"].duplicated().sum())
print("Duplicate TMDB IDs:",
      tmdb_mapping["tmdb_id"].duplicated().sum())

# Final assertion
assert len(tmdb_mapping) == 130
assert tmdb_mapping["movie_record_id"].nunique() == 130
assert tmdb_mapping["tmdb_id"].nunique() == 130
assert tmdb_mapping["tmdb_id"].isna().sum() == 0

print("\n✅ TMDB mapping validation PASSED.")

In [ ]:
# ============================================================
# TMDB API Request Setup
# ============================================================

import os
import time
import requests
import pandas as pd

from config import (
    TMDB_API_KEY,
    BASE_URL,
    LANGUAGE,
    REQUEST_DELAY,
    MAX_RETRIES,
    TIMEOUT
)

# ------------------------------------------------------------
# API session
# ------------------------------------------------------------

session = requests.Session()

session.headers.update({
    "Accept": "application/json"
})

# ------------------------------------------------------------
# TMDB movie metadata endpoint
# ------------------------------------------------------------

MOVIE_ENDPOINT = f"{BASE_URL}/movie"

print("TMDB API setup complete.")
print("Endpoint:", MOVIE_ENDPOINT)
print("Language:", LANGUAGE)
print("Request delay:", REQUEST_DELAY)
print("Max retries:", MAX_RETRIES)
print("Timeout:", TIMEOUT)

# ------------------------------------------------------------
# Verify API key
# ------------------------------------------------------------

if not TMDB_API_KEY:
    raise ValueError("TMDB_API_KEY is missing.")

print("TMDB API key available: True")

In [ ]:
# ============================================================
# TMDB Movie Metadata Fetch Function
# ============================================================

def fetch_tmdb_movie(tmdb_id):
    """
    Fetch complete movie metadata from TMDB using movie ID.

    Returns:
        dict  -> TMDB metadata
        None  -> if request ultimately fails
    """

    url = f"{MOVIE_ENDPOINT}/{int(tmdb_id)}"

    params = {
        "api_key": TMDB_API_KEY,
        "language": LANGUAGE
    }

    for attempt in range(1, MAX_RETRIES + 1):

        try:
            response = session.get(
                url,
                params=params,
                timeout=TIMEOUT
            )

            # ------------------------------------------------
            # Successful response
            # ------------------------------------------------
            if response.status_code == 200:
                return response.json()

            # ------------------------------------------------
            # Rate limit
            # ------------------------------------------------
            elif response.status_code == 429:
                wait_time = 2 ** attempt

                print(
                    f"Rate limited for TMDB ID {tmdb_id}. "
                    f"Waiting {wait_time}s..."
                )

                time.sleep(wait_time)

            # ------------------------------------------------
            # Movie not found
            # ------------------------------------------------
            elif response.status_code == 404:
                print(
                    f"TMDB ID {tmdb_id}: Movie not found (404)"
                )
                return None

            # ------------------------------------------------
            # Authentication error
            # ------------------------------------------------
            elif response.status_code in [401, 403]:
                raise RuntimeError(
                    f"TMDB authentication/permission error "
                    f"(HTTP {response.status_code})"
                )

            # ------------------------------------------------
            # Other HTTP errors
            # ------------------------------------------------
            else:
                print(
                    f"TMDB ID {tmdb_id}: "
                    f"HTTP {response.status_code} "
                    f"(attempt {attempt}/{MAX_RETRIES})"
                )

        except requests.exceptions.Timeout:
            print(
                f"TMDB ID {tmdb_id}: Request timeout "
                f"(attempt {attempt}/{MAX_RETRIES})"
            )

        except requests.exceptions.RequestException as e:
            print(
                f"TMDB ID {tmdb_id}: Request error "
                f"(attempt {attempt}/{MAX_RETRIES}) — {e}"
            )

        # ----------------------------------------------------
        # Retry delay
        # ----------------------------------------------------
        if attempt < MAX_RETRIES:
            time.sleep(2 ** (attempt - 1))

    print(
        f"TMDB ID {tmdb_id}: FAILED after "
        f"{MAX_RETRIES} attempts"
    )

    return None


print("Metadata fetch function defined successfully.")

In [ ]:
# ============================================================
# Test TMDB Metadata Fetch
# ============================================================

test_tmdb_id = 16987  # Kaho Naa... Pyaar Hai

test_movie = fetch_tmdb_movie(test_tmdb_id)

if test_movie is None:
    raise RuntimeError("TMDB test request failed.")

print("TMDB test request successful.")
print("TMDB ID:", test_movie.get("id"))
print("Title:", test_movie.get("title"))
print("Original Title:", test_movie.get("original_title"))
print("Release Date:", test_movie.get("release_date"))
print("Original Language:", test_movie.get("original_language"))
print("Runtime:", test_movie.get("runtime"))
print("Genres:", test_movie.get("genres"))
print("Production Companies:",
      test_movie.get("production_companies"))
print("Production Countries:",
      test_movie.get("production_countries"))

In [ ]:
# ============================================================
# Collect TMDB Metadata for All 130 Movies
# ============================================================

metadata_records = []
failed_tmdb_ids = []

total_movies = len(tmdb_mapping)

print(f"Starting TMDB metadata collection for {total_movies} movies...\n")

for index, row in tmdb_mapping.iterrows():

    movie_record_id = row["movie_record_id"]
    tmdb_id = int(row["tmdb_id"])

    print(
        f"[{index + 1}/{total_movies}] "
        f"{row['dataset_title']} "
        f"(TMDB ID: {tmdb_id})"
    )

    # Fetch metadata
    movie_data = fetch_tmdb_movie(tmdb_id)

    if movie_data is None:
        failed_tmdb_ids.append(tmdb_id)
        continue

    # --------------------------------------------------------
    # Store raw TMDB response
    # --------------------------------------------------------

    metadata_records.append({
        "movie_record_id": movie_record_id,
        "dataset_title": row["dataset_title"],
        "dataset_year": row["dataset_year"],
        "dataset_language": row["dataset_language"],

        "tmdb_id": movie_data.get("id"),
        "tmdb_title": movie_data.get("title"),
        "tmdb_original_title": movie_data.get("original_title"),
        "tmdb_release_date": movie_data.get("release_date"),
        "tmdb_original_language": movie_data.get("original_language"),

        "adult": movie_data.get("adult"),
        "backdrop_path": movie_data.get("backdrop_path"),
        "poster_path": movie_data.get("poster_path"),

        "budget": movie_data.get("budget"),
        "revenue": movie_data.get("revenue"),
        "runtime": movie_data.get("runtime"),

        "status": movie_data.get("status"),
        "tagline": movie_data.get("tagline"),
        "overview": movie_data.get("overview"),

        "popularity": movie_data.get("popularity"),
        "vote_average": movie_data.get("vote_average"),
        "vote_count": movie_data.get("vote_count"),

        "homepage": movie_data.get("homepage"),
        "imdb_id": movie_data.get("imdb_id"),

        "genres": movie_data.get("genres"),
        "production_companies": movie_data.get("production_companies"),
        "production_countries": movie_data.get("production_countries"),
        "spoken_languages": movie_data.get("spoken_languages"),

        "raw_tmdb_response": movie_data
    })

    # --------------------------------------------------------
    # Respect API request delay
    # --------------------------------------------------------

    time.sleep(REQUEST_DELAY)


# ------------------------------------------------------------
# Create DataFrame
# ------------------------------------------------------------

tmdb_metadata_df = pd.DataFrame(metadata_records)

print("\n" + "=" * 60)
print("TMDB METADATA COLLECTION COMPLETE")
print("=" * 60)

print("Expected movies:", total_movies)
print("Successfully collected:", len(tmdb_metadata_df))
print("Failed:", len(failed_tmdb_ids))

if failed_tmdb_ids:
    print("\nFailed TMDB IDs:")
    print(failed_tmdb_ids)
else:
    print("\n✅ No TMDB requests failed.")

In [ ]:
# ============================================================
# Validate TMDB Metadata Collection
# ============================================================

print("=" * 60)
print("TMDB METADATA VALIDATION")
print("=" * 60)

print("Rows:", len(tmdb_metadata_df))
print("Columns:", len(tmdb_metadata_df.columns))

print("\nUnique movie_record_id:",
      tmdb_metadata_df["movie_record_id"].nunique())

print("Unique tmdb_id:",
      tmdb_metadata_df["tmdb_id"].nunique())

print("Missing tmdb_id:",
      tmdb_metadata_df["tmdb_id"].isna().sum())

print("Duplicate movie_record_id:",
      tmdb_metadata_df["movie_record_id"].duplicated().sum())

print("Duplicate tmdb_id:",
      tmdb_metadata_df["tmdb_id"].duplicated().sum())

# ------------------------------------------------------------
# Compare against original verified mapping
# ------------------------------------------------------------

original_movie_ids = set(
    tmdb_mapping["movie_record_id"]
)

collected_movie_ids = set(
    tmdb_metadata_df["movie_record_id"]
)

missing_movie_ids = original_movie_ids - collected_movie_ids
extra_movie_ids = collected_movie_ids - original_movie_ids

print("\nMissing movie records:", missing_movie_ids)
print("Extra movie records:", extra_movie_ids)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert len(tmdb_metadata_df) == 130
assert tmdb_metadata_df["movie_record_id"].nunique() == 130
assert tmdb_metadata_df["tmdb_id"].nunique() == 130
assert tmdb_metadata_df["tmdb_id"].isna().sum() == 0
assert tmdb_metadata_df["movie_record_id"].duplicated().sum() == 0
assert tmdb_metadata_df["tmdb_id"].duplicated().sum() == 0
assert len(missing_movie_ids) == 0
assert len(extra_movie_ids) == 0

print("\n✅ TMDB METADATA VALIDATION PASSED")

In [ ]:
# ============================================================
# Save Raw TMDB Metadata
# ============================================================

OUTPUT_PATH = os.path.join(
    PROJECT_ROOT,
    "01_data",
    "raw",
    "ICMB_130_TMDB_METADATA.csv"
)

# Save metadata
tmdb_metadata_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Verify file was created
print("TMDB metadata saved successfully.")
print("Output path:", OUTPUT_PATH)
print("File exists:", os.path.exists(OUTPUT_PATH))

# Check file size
if os.path.exists(OUTPUT_PATH):
    file_size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
    print(f"File size: {file_size_mb:.2f} MB")

In [ ]:
# ============================================================
# CELL 9 — Verify Saved TMDB Metadata
# ============================================================

saved_metadata = pd.read_csv(OUTPUT_PATH)

print("=" * 60)
print("SAVED TMDB METADATA VERIFICATION")
print("=" * 60)

print("Rows:", len(saved_metadata))
print("Columns:", len(saved_metadata.columns))
print("Unique movie_record_id:",
      saved_metadata["movie_record_id"].nunique())
print("Unique tmdb_id:",
      saved_metadata["tmdb_id"].nunique())
print("Missing tmdb_id:",
      saved_metadata["tmdb_id"].isna().sum())
print("Duplicate movie_record_id:",
      saved_metadata["movie_record_id"].duplicated().sum())
print("Duplicate tmdb_id:",
      saved_metadata["tmdb_id"].duplicated().sum())

# Final checks
assert len(saved_metadata) == 130
assert saved_metadata["movie_record_id"].nunique() == 130
assert saved_metadata["tmdb_id"].nunique() == 130
assert saved_metadata["tmdb_id"].isna().sum() == 0
assert saved_metadata["movie_record_id"].duplicated().sum() == 0
assert saved_metadata["tmdb_id"].duplicated().sum() == 0

print("\n✅ SAVED FILE VERIFICATION PASSED")
print("TMDB metadata collection stage is complete.")